# Solution 5 (local) — Generation and faithfulness, no API

Same experiment as the API version, but generation **and** judging run on the Colab GPU. No rate limits, no quota, and reproducible by anyone with a GPU.

| | Condition |
|---|---|
| **C1** | MSA query, base encoder (ceiling) |
| **C2** | Darija query, base encoder (mismatch) |
| **C3** | Darija query, fine-tuned encoder |
| **C4** | Gold passage supplied directly (oracle) |

**Efficiency:** prompts are **batched** (8 at a time), and faithfulness + correctness are judged in a **single call** — half the work of the API version. Encoders are freed before the LLM loads so both never occupy VRAM together.

**Model:** `Qwen2.5-7B-Instruct` in 4-bit (fits a free T4). Swap to `MBZUAI-Paris/Atlas-Chat-9B` in Cell 2 for Darija-specialised generation.

**Upload:** `corpus_v2.json`, `qa_pairs_wiki.json`, `finetuned-dialect-encoder/`. **GPU required.**

### Install

In [ ]:
 !pip install -q faiss-cpu rank_bm25 sentence-transformers transformers accelerate bitsandbytes

### Config

In [ ]:
CONFIG = {
    "corpus_file": "corpus_v2.json",
    "wiki_qa_file": "qa_pairs_wiki.json",
    "base_encoder": "intfloat/multilingual-e5-base",
    "finetuned_path": "finetuned-dialect-encoder",   # None -> skip C3
    "alpha": 0.8,
    "top_k": 5,

    "n_eval": 100,
    # Alternatives: "MBZUAI-Paris/Atlas-Chat-9B" (Darija-specialised),
    #               "Qwen/Qwen2.5-3B-Instruct" (faster, weaker)
    "llm": "Qwen/Qwen2.5-7B-Instruct",
    "load_4bit": True,
    "max_new_tokens": 128,      # answers are one short sentence; judges output 1 word
    "batch_size": 8,            # generation is batched; raise if VRAM allows

    "checkpoint": "local_gen_checkpoint.json",
    "seed": 42,
}

### Load data

In [ ]:
import json, random, re, os, gc
import numpy as np
import pandas as pd

with open(CONFIG["corpus_file"], encoding="utf-8") as f:
    corpus = json.load(f)
with open(CONFIG["wiki_qa_file"], encoding="utf-8") as f:
    wiki_qa = json.load(f)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)

qa = [q for q in wiki_qa if q["source_chunk_id"] in known]
random.Random(CONFIG["seed"]).shuffle(qa)
eval_qa = qa[: CONFIG["n_eval"]]
print(f"Corpus {len(corpus)} | QA {len(qa)} | evaluating {len(eval_qa)}")

### BM25

In [ ]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t)
    t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t)
    t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t)
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
_bm = {}
def bm25_scores(q):
    if q not in _bm:
        _bm[q] = np.asarray(bm25.get_scores(tokenize(q)))
    return _bm[q]

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

### Retrain the dialect encoder (run before building contexts)

In [ ]:
# Retrain the dialect encoder.
# MUST run BEFORE the contexts are built: it redefines eval_qa, and the context
# maps are keyed by question id. Building contexts first and retraining second
# is what caused the earlier KeyError.

from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import random, torch, gc

rnd = random.Random(CONFIG["seed"])
passages = sorted({q["source_chunk_id"] for q in qa})
rnd.shuffle(passages)
test_p = set(passages[: int(len(passages) * 0.35)])

# Evaluation questions come only from held-out passages, so the generation
# results are never computed on data the encoder trained on.
train_qa = [q for q in qa if q["source_chunk_id"] not in test_p]
eval_qa  = [q for q in qa if q["source_chunk_id"] in test_p][: CONFIG["n_eval"]]
assert not ({q["source_chunk_id"] for q in train_qa} & {q["source_chunk_id"] for q in eval_qa})
print(f"Retrain on {len(train_qa)} questions | evaluate on {len(eval_qa)}")

examples = []
for q in train_qa:
    examples.append(InputExample(texts=[f'query: {q["darija_query"]}',
                                        f'passage: {corpus_map[q["source_chunk_id"]]}']))
    examples.append(InputExample(texts=[f'query: {q["darija_query"]}',
                                        f'query: {q["msa_query"]}']))
rnd.shuffle(examples)

m = SentenceTransformer(CONFIG["base_encoder"])
loader = DataLoader(examples, shuffle=True, batch_size=16)
steps = len(loader) * 5
m.fit(train_objectives=[(loader, losses.MultipleNegativesRankingLoss(m))],
      epochs=5, warmup_steps=int(steps * 0.1),
      optimizer_params={"lr": 2e-5}, show_progress_bar=True)
m.save("finetuned-dialect-encoder")
del m
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Saved finetuned-dialect-encoder")

# Any checkpoint from a previous run refers to the old question set.
import os
if os.path.exists(CONFIG["checkpoint"]):
    os.remove(CONFIG["checkpoint"])
    print("Removed stale checkpoint (question set changed).")

### Build contexts for all four conditions, then free the encoders

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

def build(path):
    m = SentenceTransformer(path)
    e = m.encode([f"passage: {t}" for t in corpus_texts],
                 normalize_embeddings=True, batch_size=32, show_progress_bar=True)
    return m, np.asarray(e, "float32")

def retrieve_ids(model, emb, query, k):
    q = model.encode([f"query: {query}"], normalize_embeddings=True)[0]
    s = CONFIG["alpha"] * minmax(emb @ q) + (1 - CONFIG["alpha"]) * minmax(bm25_scores(query))
    return [corpus_ids[i] for i in np.argsort(-s)[:k]]

contexts, K = {}, CONFIG["top_k"]

bm, be = build(CONFIG["base_encoder"])
contexts["C1_msa_base"]    = {q["id"]: retrieve_ids(bm, be, q["msa_query"], K) for q in eval_qa}
contexts["C2_darija_base"] = {q["id"]: retrieve_ids(bm, be, q["darija_query"], K) for q in eval_qa}
del bm, be
gc.collect(); torch.cuda.empty_cache()

if CONFIG["finetuned_path"] and os.path.exists(CONFIG["finetuned_path"]):
    fm, fe = build(CONFIG["finetuned_path"])
    contexts["C3_darija_finetuned"] = {q["id"]: retrieve_ids(fm, fe, q["darija_query"], K) for q in eval_qa}
    del fm, fe
    gc.collect(); torch.cuda.empty_cache()
else:
    print("Fine-tuned encoder not found — skipping C3.")

contexts["C4_oracle"] = {q["id"]: [q["source_chunk_id"]] for q in eval_qa}

# Encoders are released before the LLM loads, so both never sit in VRAM together.
for cond, d in contexts.items():
    hit = sum(1 for q in eval_qa if q["source_chunk_id"] in d[q["id"]]) / len(eval_qa)
    print(f"  {cond:<22} gold in context: {hit:.1%}")

json.dump({c: d for c, d in contexts.items()},
          open("contexts.json", "w", encoding="utf-8"), ensure_ascii=False)

### Load the local LLM

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
) if CONFIG["load_4bit"] else None

tok = AutoTokenizer.from_pretrained(CONFIG["llm"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"   # required for correct batched generation

llm = AutoModelForCausalLM.from_pretrained(
    CONFIG["llm"], quantization_config=quant,
    device_map="auto", torch_dtype=torch.float16,
)
llm.eval()
print(f"Loaded {CONFIG['llm']}")
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — this will be slow")

@torch.no_grad()
def chat_batch(prompts, max_new_tokens=None):
    """Batched chat completion. Batching is what makes local generation
    competitive with an API here."""
    mnt = max_new_tokens or CONFIG["max_new_tokens"]
    texts = [tok.apply_chat_template([{"role": "user", "content": p}],
                                     tokenize=False, add_generation_prompt=True)
             for p in prompts]
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True,
              max_length=3072).to(llm.device)
    out = llm.generate(**enc, max_new_tokens=mnt, do_sample=False,
                       pad_token_id=tok.pad_token_id)
    gen = out[:, enc["input_ids"].shape[1]:]
    return [tok.decode(g, skip_special_tokens=True).strip() for g in gen]

# quick smoke test
print("Smoke test:", chat_batch(["أجب بكلمة واحدة: ما عاصمة المغرب؟"], 20)[0])

### Prompts

In [ ]:
GEN_PROMPT = """أجب عن السؤال التالي اعتمادا فقط على النصوص المرفقة.
إذا لم تكن الإجابة موجودة في النصوص، قل بالضبط: المعلومة غير متوفرة في النصوص
لا تستعمل أي معرفة خارجية. أجب بجملة واحدة قصيرة فقط.

النصوص:
{context}

السؤال: {question}

الإجابة:"""

# Faithfulness and correctness are judged in ONE call to halve the work.
# They are separate constructs: an answer can faithfully report a passage that
# retrieval wrongly supplied, and so be faithful but incorrect.
JUDGE_PROMPT = """النصوص المرجعية:
{context}

السؤال: {question}
الإجابة الصحيحة: {gold}
الإجابة المقدمة: {answer}

أجب عن سؤالين بدقة:
1. هل كل ما ورد في الإجابة المقدمة مدعوم صراحة بالنصوص المرجعية؟
2. هل الإجابة المقدمة مطابقة في المعنى للإجابة الصحيحة؟ اختلاف الصياغة مقبول، أما اختلاف الأرقام أو الأسماء أو التواريخ فغير مقبول.

أجب بهذا الشكل فقط وبدون أي شرح:
مدعوم: نعم/لا
مطابق: نعم/لا"""

def ctx_text(chunk_ids):
    return "\n\n".join(f"[{i+1}] {corpus_map[c]}" for i, c in enumerate(chunk_ids))

def parse_judge(text):
    t = (text or "").replace("،", " ")
    faithful = correct = 0
    for line in t.split("\n"):
        if "مدعوم" in line:
            faithful = 1 if "نعم" in line else 0
        elif "مطابق" in line:
            correct = 1 if "نعم" in line else 0
    return faithful, correct

### Run (batched + checkpointed)

In [ ]:
from tqdm.auto import tqdm

records = []
if os.path.exists(CONFIG["checkpoint"]):
    records = json.load(open(CONFIG["checkpoint"], encoding="utf-8"))
    print(f"Resuming with {len(records)} records.")
done = {(r["qid"], r["condition"]) for r in records}

CONDITION_QUERY = {
    "C1_msa_base": "msa_query",
    "C2_darija_base": "darija_query",
    "C3_darija_finetuned": "darija_query",
    "C4_oracle": "darija_query",
}
byid = {q["id"]: q for q in eval_qa}
B = CONFIG["batch_size"]

for cond, ctx_map in contexts.items():
    qfield = CONDITION_QUERY[cond]
    todo = [q for q in eval_qa if (q["id"], cond) not in done]
    if not todo:
        continue
    print(f"\n=== {cond} ({len(todo)} to do) ===")

    for i in tqdm(range(0, len(todo), B)):
        batch = todo[i:i + B]
        chunks = [ctx_map[q["id"]] for q in batch]

        answers = chat_batch([GEN_PROMPT.format(context=ctx_text(c), question=q[qfield])
                              for q, c in zip(batch, chunks)])

        verdicts = chat_batch([JUDGE_PROMPT.format(context=ctx_text(c),
                                                   question=q["msa_query"],
                                                   gold=q["gold_answer"],
                                                   answer=a)
                               for q, c, a in zip(batch, chunks, answers)], 40)

        for q, c, a, v in zip(batch, chunks, answers, verdicts):
            f, ok = parse_judge(v)
            records.append({
                "qid": q["id"], "condition": cond,
                "gold_in_context": int(q["source_chunk_id"] in c),
                "answer": a, "faithful": f, "correct": ok,
                "refused": int("غير متوفرة" in (a or "")),
            })

        json.dump(records, open(CONFIG["checkpoint"], "w", encoding="utf-8"),
                  ensure_ascii=False, indent=2)

gen = pd.DataFrame(records)
gen.to_csv("generation_local_raw.csv", index=False)
print(f"\nComplete: {len(gen)} generations.")

### Results by condition

In [ ]:
print("=" * 88)
print("GENERATION RESULTS BY CONDITION")
print("=" * 88)
order = [c for c in CONDITION_QUERY if c in gen.condition.unique()]
summary = gen.groupby("condition").agg(
    n=("qid", "count"),
    gold_in_context=("gold_in_context", "mean"),
    faithfulness=("faithful", "mean"),
    correctness=("correct", "mean"),
    refusal_rate=("refused", "mean"),
).reindex(order)
print(summary.to_string(float_format=lambda x: f"{x:.3f}"))
summary.to_csv("generation_local_summary.csv")

print("""
  gold_in_context  how often retrieval put the right passage before the generator
  faithfulness     answer supported by the context it was actually given
  correctness      answer matches the gold answer   <- what users care about
  refusal_rate     model declared the information absent
""")

### Paired comparisons with bootstrap CIs

In [ ]:
rng = np.random.default_rng(CONFIG["seed"])

def paired(a_cond, b_cond, col):
    a = gen[gen.condition == a_cond].set_index("qid")[col]
    b = gen[gen.condition == b_cond].set_index("qid")[col]
    common = a.index.intersection(b.index)
    d = (a.loc[common] - b.loc[common]).values.astype(float)
    idx = rng.integers(0, len(d), size=(1000, len(d)))
    m = d[idx].mean(axis=1)
    lo, hi = np.percentile(m, [2.5, 97.5])
    return d.mean(), lo, hi

print("=" * 88)
print("PAIRED COMPARISONS (95% CI)")
print("=" * 88)
rows = []
for a, b, label in [
    ("C1_msa_base", "C2_darija_base", "Does the dialect gap reach the answers?"),
    ("C3_darija_finetuned", "C2_darija_base", "Does the fine-tuned retriever help answers?"),
    ("C4_oracle", "C2_darija_base", "How much is lost to retrieval vs generation?"),
]:
    if a not in gen.condition.unique() or b not in gen.condition.unique():
        continue
    print(f"\n{label}   [{a} - {b}]")
    for col in ["correct", "faithful"]:
        d, lo, hi = paired(a, b, col)
        sig = "yes" if (lo > 0 or hi < 0) else "no"
        print(f"  {col:<10} {d:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  significant: {sig}")
        rows.append({"comparison": f"{a} - {b}", "metric": col,
                     "diff": d, "lo": lo, "hi": hi, "significant": sig})
pd.DataFrame(rows).to_csv("generation_local_comparisons.csv", index=False)

### When retrieval misses, does the generator refuse or hallucinate?

In [ ]:
print("\n" + "=" * 88)
print("WHEN RETRIEVAL FAILS, WHAT DOES THE GENERATOR DO?")
print("=" * 88)
d2 = gen[gen.condition == "C2_darija_base"]
miss, hit = d2[d2.gold_in_context == 0], d2[d2.gold_in_context == 1]
if len(miss):
    for label, sub in [("MISSING", miss), ("PRESENT", hit)]:
        print(f"\nGold passage {label} ({len(sub)} cases):")
        print(f"  correctness   {sub.correct.mean():.3f}")
        print(f"  faithfulness  {sub.faithful.mean():.3f}")
        print(f"  refusal rate  {sub.refused.mean():.3f}")
    print("""
Low refusal + low correctness on the MISSING rows is the failure the proposal
predicted: the generator does not notice the context is wrong and answers
confidently anyway. A high refusal rate means it degrades safely instead.""")
else:
    print("Retrieval never missed in this sample — raise n_eval for this analysis.")

### Example failures for the paper

In [ ]:
print("\n=== Sample failures (dialect condition, incorrect answer) ===\n")
for _, r in gen[(gen.condition == "C2_darija_base") & (gen.correct == 0)].head(5).iterrows():
    q = byid[r["qid"]]
    print(f"Q (darija): {q['darija_query']}")
    print(f"Gold      : {q['gold_answer']}")
    print(f"Generated : {str(r['answer'])[:200]}")
    print(f"gold in context: {bool(r['gold_in_context'])} | faithful: {bool(r['faithful'])}")
    print("-" * 80)

from google.colab import files
files.download("generation_local_raw.csv")
files.download("generation_local_summary.csv")
files.download("generation_local_comparisons.csv")